### Scott 10K classifiers and 1v1 - local machine

##### An auxillary notebook being used to run some other AD-CN classification tasks on my local machine

#### Goal: run more focused version of scott_10k_classifiers and scott_10k_1v1

#### (Also in Scott 10K classifiers and Scott 10K LM classifiers) assess whether Desikan LR and XGB performance is similar via paired tests

#### Scott 10k 1v1 - Use a common dataset - ensures that differences in ML performance aren't due to differences in sample size

In [5]:
import pandas as pd
import numpy as np
import os
import re
import scipy
from scipy.stats import wilcoxon, ttest_rel, levene, kstest
import pingouin as pg

root_path = 'C:/Users/User/Downloads/deconstructalz.github.io/scott_10k_analysis/classifiers/'

desikan_df = pd.read_csv(os.path.join(root_path, 'fold_metrics_desikan.csv'), low_memory = False)
destrieux_df = pd.read_csv(os.path.join(root_path, 'fold_metrics_destrieux.csv'), low_memory = False)
combined_df = desikan_df.join(destrieux_df)

accuracies = combined_df[['Desikan LR accuracy', 'Desikan XGB accuracy']]
f1s = combined_df[['Desikan LR F1', 'Desikan XGB F1']]
sensitivities = combined_df[['Desikan LR Sensitivity', 'Desikan XGB Sensitivity']]
aucprs = combined_df[['Desikan LR AUC-PR', 'Desikan XGB AUC-PR']]

thresh = len([accuracies, f1s, sensitivities, aucprs])

for n in [accuracies, f1s, sensitivities, aucprs]:
    print(n)
    print(n.values[:,0], n.values[:,1])
    _, p_levene = levene(n.values[:,0], n.values[:,1])
    equal_var = p_levene > 0.05

    # 2. Kolmogorov-Smirnov Test for Normality (standardized)
    is_normal = True
    for g in [n.values[:,0], n.values[:,1]]:
        standardised_group = (g - g.mean()) / g.std()
        _, p_ks = kstest(standardised_group, 'norm')
        if p_ks < 0.05:
            is_normal = False
            break

    # 3. Pairwise test
    if is_normal and equal_var:
        test_name = "Paired t-test"
        result = ttest_rel(n.values[:,0], n.values[:,1])
    elif is_normal and not equal_var:
        test_name = "Paired t-test"
        result = ttest_rel(n.values[:,0], n.values[:,1])
    else:
        test_name = "Wilcoxon signed-rank test"
        result = wilcoxon(n.values[:,0], n.values[:,1])

    print(result, test_name, is_normal, equal_var)
    print(f'Raw p-value = {result.pvalue}. Adjusted by n = 4: {float(result.pvalue * thresh)}')
    print(bool(float(result.pvalue) < 0.05))# Prints adjusted p value. Threshold is 4 - for each set of metrics
    print(bool(float(result.pvalue / thresh) < 0.05))# Prints adjusted p value. Threshold is 4 - for each set of metrics
    hedges_g = pg.compute_effsize(n.values[:,0], n.values[:,1], paired = True, eftype='hedges')
    print(f"Hedges' g: {hedges_g}")
    print(' ')

    ## Focus on the AUC-PR statistic, since that is arguably the most holistic


   Desikan LR accuracy  Desikan XGB accuracy
0             0.875546              0.870451
1             0.864629              0.844250
2             0.862445              0.838428
3             0.882096              0.859534
4             0.838428              0.831150
[0.87554585 0.86462882 0.86244541 0.88209607 0.83842795] [0.87045124 0.84425036 0.83842795 0.85953421 0.83114993]
TtestResult(statistic=np.float64(3.9577531696172086), pvalue=np.float64(0.016709059917068222), df=np.int64(4)) Paired t-test True True
Raw p-value = 0.016709059917068222. Adjusted by n = 4: 0.06683623966827289
True
True
Hedges' g: 0.8763380135619994
 
   Desikan LR F1  Desikan XGB F1
0       0.846361        0.838475
1       0.859091        0.834365
2       0.833186        0.797075
3       0.845420        0.817062
4       0.782353        0.776493
[0.84636119 0.85909091 0.83318623 0.84541985 0.78235294] [0.8384755  0.83436533 0.79707495 0.81706161 0.77649326]
TtestResult(statistic=np.float64(3.4881541031934424)

In [6]:
import pandas as pd
import numpy as np
import os
import re
import scipy
from scipy.stats import wilcoxon, ttest_rel, levene, kstest

root_path = 'C:/Users/User/Downloads/deconstructalz.github.io/scott_10k_analysis/classifiers/'

desikan_df = pd.read_csv(os.path.join(root_path, 'k_10_fold_metrics_desikan.csv'), low_memory = False)
destrieux_df = pd.read_csv(os.path.join(root_path, 'k_10_fold_metrics_destrieux.csv'), low_memory = False)
combined_df = desikan_df.join(destrieux_df)

accuracies = combined_df[['Desikan LR accuracy', 'Desikan XGB accuracy', 'Destrieux LR accuracy', 'Destrieux XGB accuracy']]
f1s = combined_df[['Desikan LR F1', 'Desikan XGB F1', 'Destrieux LR F1', 'Destrieux XGB F1']]
sensitivities = combined_df[['Desikan LR Sensitivity', 'Desikan XGB Sensitivity', 'Destrieux LR Sensitivity', 'Destrieux XGB Sensitivity']]
aucprs = combined_df[['Desikan LR AUC-PR', 'Desikan XGB AUC-PR', 'Destrieux LR AUC-PR', 'Destrieux XGB AUC-PR']]

thresh = len([accuracies, f1s, sensitivities, aucprs])

for n in [accuracies, f1s, sensitivities, aucprs]:
    n_mean = n.mean()
    n_std = n.std()
    # print(n_mean)
    # print(n_std)
    print(n_mean.idxmax(), n_mean.max())

    print('Mean')
    print(n.mean().round(decimals = 2))
    print('Stdev')
    print(n.std().round(decimals = 2))


Desikan LR accuracy 0.8660844250363902
Mean
Desikan LR accuracy       0.87
Desikan XGB accuracy      0.85
Destrieux LR accuracy     0.86
Destrieux XGB accuracy    0.84
dtype: float64
Stdev
Desikan LR accuracy       0.03
Desikan XGB accuracy      0.03
Destrieux LR accuracy     0.02
Destrieux XGB accuracy    0.03
dtype: float64
Desikan LR F1 0.8342644469766075
Mean
Desikan LR F1       0.83
Desikan XGB F1      0.82
Destrieux LR F1     0.82
Destrieux XGB F1    0.80
dtype: float64
Stdev
Desikan LR F1       0.04
Desikan XGB F1      0.04
Destrieux LR F1     0.03
Destrieux XGB F1    0.04
dtype: float64
Desikan LR Sensitivity 0.8321210581078702
Mean
Desikan LR Sensitivity       0.83
Desikan XGB Sensitivity      0.81
Destrieux LR Sensitivity     0.82
Destrieux XGB Sensitivity    0.79
dtype: float64
Stdev
Desikan LR Sensitivity       0.06
Desikan XGB Sensitivity      0.04
Destrieux LR Sensitivity     0.05
Destrieux XGB Sensitivity    0.05
dtype: float64
Desikan LR AUC-PR 0.9121193588617041
Mean
D

In [19]:
import pandas as pd
import numpy as np
import os
import sklearn
import shap
import matplotlib.pyplot as plt 
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.metrics import precision_recall_curve, auc, accuracy_score, f1_score, recall_score, average_precision_score
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score 
from matplotlib import font_manager

In [29]:

scaler = StandardScaler()

seed = 42

kf = KFold()

lr = LogisticRegression(random_state = seed, class_weight = 'balanced')

raw_desikan_cort_thick = [col for col in df.columns.tolist() if 'aparc_thickness' in col]
raw_desikan_cort_thick_banned = ['lh_MeanThickness_thickness_aparc_thickness_lh', 'BrainSegVolNotVent_aparc_thickness_lh', 
                                 'eTIV_aparc_thickness_lh', 'rh_MeanThickness_thickness_aparc_thickness_rh', 
                                 'BrainSegVolNotVent_aparc_thickness_rh', 'eTIV_aparc_thickness_rh']
raw_desikan_roi_gmv = [col for col in df.columns.tolist() if 'aparc_volume' in col or 'aseg_stats' in col]
raw_desikan_roi_gmv_banned = list(set(['Right-Cerebellum-White-Matter_aseg_stats', 'Brain-Stem_aseg_stats', 
                                       'CSF_aseg_stats','Left-vessel_aseg_stats', 'Left-choroid-plexus_aseg_stats',
                                       'Right-Lateral-Ventricle_aseg_stats' ,'Right-Inf-Lat-Vent_aseg_stats',
                                       '3rd-Ventricle_aseg_stats','4th-Ventricle_aseg_stats',
                                       'BrainSegVolNotVent_aparc_volume_lh', 'BrainSegVolNotVent_aparc_volume_rh', 
                                       'Left-Lateral-Ventricle_aseg_stats', 'Left-Inf-Lat-Vent_aseg_stats', 
                                       'Left-Cerebellum-White-Matter_aseg_stats', 'Right-vessel_aseg_stats', 
                                       'Right-choroid-plexus_aseg_stats', '5th-Ventricle_aseg_stats', 
                                       'WM-hypointensities_aseg_stats', 'Left-WM-hypointensities_aseg_stats', 
                                       'Right-WM-hypointensities_aseg_stats', 'non-WM-hypointensities_aseg_stats', 
                                       'Left-non-WM-hypointensities_aseg_stats', 'Right-non-WM-hypointensities_aseg_stats', 
                                       'Optic-Chiasm_aseg_stats', 'CC_Posterior_aseg_stats', 'CC_Mid_Posterior_aseg_stats', 
                                       'CC_Central_aseg_stats', 'CC_Mid_Anterior_aseg_stats', 'CC_Anterior_aseg_stats', 
                                       'BrainSegVol_aseg_stats', 'BrainSegVolNotVent_aseg_stats', 'lhCortexVol_aseg_stats', 
                                       'rhCortexVol_aseg_stats', 'CortexVol_aseg_stats', 'lhCerebralWhiteMatterVol_aseg_stats', 
                                       'rhCerebralWhiteMatterVol_aseg_stats', 'CerebralWhiteMatterVol_aseg_stats', 'SubCortGrayVol_aseg_stats', 
                                       'TotalGrayVol_aseg_stats', 'SupraTentorialVol_aseg_stats', 'SupraTentorialVolNotVent_aseg_stats', 
                                       'MaskVol_aseg_stats', 'BrainSegVol-to-eTIV_aseg_stats', 'MaskVol-to-eTIV_aseg_stats', 'lhSurfaceHoles_aseg_stats', 
                                       'rhSurfaceHoles_aseg_stats', 'SurfaceHoles_aseg_stats']))


column_dicts = {
    'desikan_coeffs_fs': ['desikan_D1', 'desikan_NET', 'desikan_5HT1A', 'desikan_5HT2A',
     'desikan_5HT4', 'desikan_5HT6', 'desikan_a4b2', 'desikan_M1', 'desikan_vAChT', 'desikan_NMDA', 
     'desikan_mGluR5', 'desikan_GABAA/BZ', 'desikan_H3', 'desikan_CB1', 'desikan_MOR'],
    'desikan_vbm' : [col for col in df.columns.tolist() if 'desikan_vbm' in col],
    'desikan_cort_thick' : [col for col in raw_desikan_cort_thick if col not in raw_desikan_cort_thick_banned],
    'desikan_roi_gmv' :  [col for col in raw_desikan_roi_gmv if col not in raw_desikan_roi_gmv_banned]}

var_names = {
    'desikan_coeffs_fs': 'RFECV_COEFFS',
    'desikan_vbm' : 'VBM',
    'desikan_cort_thick' : 'SBM_CORT_THICK',
    'desikan_roi_gmv' :  'SBM_ROI_GMV'}


df = pd.read_csv('C:/Users/User/Downloads/deconstructalz.github.io/scott_10k_housekeeping/sbm_w_vbm_scott10k_alliedhealth.csv', low_memory = False)
df = df[(df['DIAGNOSIS'] == 1) | (df['DIAGNOSIS'] == 3)]
print(df.shape)
superlistofvars = []
for var in column_dicts.values():
    for col in var:
        superlistofvars.append(col)
print(len(superlistofvars))


desikan_df = df[['DIAGNOSIS'] + superlistofvars].dropna()
desikan_df = desikan_df.apply(pd.to_numeric, errors = 'coerce')
print(desikan_df.shape)
y = desikan_df['DIAGNOSIS'].map(lambda x: 0 if x == 1 else 1)


(6875, 928)
255
(6842, 256)


In [34]:
for key in column_dicts.keys():
    flag = var_names[key]
    X = desikan_df.drop(columns = 'DIAGNOSIS')[column_dicts[key]]
    print(f'Processing {flag} : shape of available features = {X.shape}')

    accuracies_lr = []
    f1_scores_lr = []
    sensitivities_lr = []
    aucprs_lr = []

    accuracies_xgb = []
    f1_scores_xgb = []
    sensitivities_xgb = []
    aucprs_xgb = []


    for i, (train_index, test_index) in enumerate(kf.split(X)):
        print(f"Fold {i},", end = ' ')

        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train,y_test = y.iloc[train_index], y.iloc[test_index]

        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)


        xgb_balancer = float(y[y == 0].shape[0])/ float(y[y == 1].shape[0])
        print(xgb_balancer)

        ## Instantiate xgb with the balancer - mitigates class imbalance

        xgb = XGBClassifier(random_state = seed, scale_pos_weight = xgb_balancer)

        lr.fit(X_train, y_train)
        xgb.fit(X_train, y_train)

        lr_pred_diags = lr.predict(X_test)
        xgb_pred_diags = xgb.predict(X_test)

        lr_pred_probas = lr.predict_proba(X_test)[:,1]
        xgb_pred_probas = xgb.predict_proba(X_test)[:,1]

        precision_lr, recall_lr, thresholds_lr = precision_recall_curve(y_test, lr_pred_probas)
        precision_xgb, recall_xgb, thresholds_xgb = precision_recall_curve(y_test, xgb_pred_probas)

        aucpr_lr = auc(recall_lr, precision_lr)
        aucprs_lr.append(aucpr_lr)
        aucpr_xgb = auc(recall_xgb, precision_xgb)
        aucprs_xgb.append(aucpr_xgb)

        acc_lr = accuracy_score(y_test, lr_pred_diags)
        accuracies_lr.append(acc_lr)
        acc_xgb = accuracy_score(y_test, xgb_pred_diags)
        accuracies_xgb.append(acc_xgb)

        f1_lr = f1_score(y_test, lr_pred_diags)
        f1_scores_lr.append(f1_lr)
        f1_xgb = f1_score(y_test, xgb_pred_diags)
        f1_scores_xgb.append(f1_xgb)

        sensitivity_lr = recall_score(y_test, lr_pred_diags)
        sensitivities_lr.append(sensitivity_lr)
        sensitivity_xgb = recall_score(y_test, xgb_pred_diags)
        sensitivities_xgb.append(sensitivity_xgb)

    flag_u = key.capitalize()

    data = {
        f"{flag_u} LR accuracy": accuracies_lr,
        f"{flag_u} XGB accuracy": accuracies_xgb,
        f"{flag_u} LR F1": f1_scores_lr,
        f"{flag_u} XGB F1": f1_scores_xgb,
        f"{flag_u} LR Sensitivity": sensitivities_lr,
        f"{flag_u} XGB Sensitivity": sensitivities_xgb,
        f"{flag_u} LR AUC-PR": aucprs_lr,
        f"{flag_u} XGB AUC-PR": aucprs_xgb,
    }
    
    metrics = pd.DataFrame(data)

    print(metrics)

    metrics.to_csv(f'C:/Users/User/Downloads/deconstructalz.github.io/scott_10k_analysis/classifiers/common_dataset_fold_metrics_{key.lower()}.csv', index = False)
        

Processing RFECV_COEFFS : shape of available features = (6842, 15)
Fold 0, 1.4340092493774457
Fold 1, 1.4340092493774457
Fold 2, 1.4340092493774457
Fold 3, 1.4340092493774457
Fold 4, 1.4340092493774457
   Desikan_coeffs_fs LR accuracy  Desikan_coeffs_fs XGB accuracy  \
0                       0.874361                        0.860482   
1                       0.868517                        0.834916   
2                       0.863304                        0.826023   
3                       0.878655                        0.864766   
4                       0.839181                        0.818713   

   Desikan_coeffs_fs LR F1  Desikan_coeffs_fs XGB F1  \
0                 0.843352                  0.825889   
1                 0.862385                  0.820919   
2                 0.833482                  0.782450   
3                 0.841603                  0.821601   
4                 0.784314                  0.760155   

   Desikan_coeffs_fs LR Sensitivity  Desikan_coeffs_

In [5]:
#1. Does LR or XGB perform better on each metric on average for each set of features?
import pandas as pd
import numpy as np
import scipy
import os
from scipy.stats import wilcoxon, ttest_rel, levene, kstest
import fnmatch

list_o_dfs = []

files = os.listdir('C:/Users/User/Downloads/deconstructalz.github.io/scott_10k_analysis/classifiers/')
subset = [f for f in files if fnmatch.fnmatch(f, 'common_dataset_fold_metrics_*.csv')]
for f in subset:
    df = pd.read_csv(os.path.join('C:/Users/User/Downloads/deconstructalz.github.io/scott_10k_analysis/classifiers/', f), low_memory = False)
    print(df.columns.tolist())
    list_o_dfs.append(df)

final_df = pd.concat(list_o_dfs, axis = 1)
final_df

accuracies = final_df[[col for col in final_df  if 'LR accuracy' in col]]
f1s = final_df[[col for col in final_df  if 'LR F1' in col]]
sensitivities = final_df[[col for col in final_df  if 'LR Sensitivity' in col]]
aucprs = final_df[[col for col in final_df  if 'LR AUC-PR'in col]]

for data in [accuracies, f1s, sensitivities, aucprs]:
    print(data.shape)
    print(data.mean().round(2))
    print(data.std().round(2))
    metrics_df = pd.concat([data.mean().round(2), data.std().round(2)], axis = 1)
    metrics_df.columns = ['Mean', 'Stdev']
    print(metrics_df)
    dmean = data.mean()
    print(dmean.idxmax(), dmean.max())

['Desikan_coeffs_fs LR accuracy', 'Desikan_coeffs_fs XGB accuracy', 'Desikan_coeffs_fs LR F1', 'Desikan_coeffs_fs XGB F1', 'Desikan_coeffs_fs LR Sensitivity', 'Desikan_coeffs_fs XGB Sensitivity', 'Desikan_coeffs_fs LR AUC-PR', 'Desikan_coeffs_fs XGB AUC-PR']
['Desikan_cort_thick LR accuracy', 'Desikan_cort_thick XGB accuracy', 'Desikan_cort_thick LR F1', 'Desikan_cort_thick XGB F1', 'Desikan_cort_thick LR Sensitivity', 'Desikan_cort_thick XGB Sensitivity', 'Desikan_cort_thick LR AUC-PR', 'Desikan_cort_thick XGB AUC-PR']
['Desikan_roi_gmv LR accuracy', 'Desikan_roi_gmv XGB accuracy', 'Desikan_roi_gmv LR F1', 'Desikan_roi_gmv XGB F1', 'Desikan_roi_gmv LR Sensitivity', 'Desikan_roi_gmv XGB Sensitivity', 'Desikan_roi_gmv LR AUC-PR', 'Desikan_roi_gmv XGB AUC-PR']
['Desikan_vbm LR accuracy', 'Desikan_vbm XGB accuracy', 'Desikan_vbm LR F1', 'Desikan_vbm XGB F1', 'Desikan_vbm LR Sensitivity', 'Desikan_vbm XGB Sensitivity', 'Desikan_vbm LR AUC-PR', 'Desikan_vbm XGB AUC-PR']
(5, 4)
Desikan_coeff

In [79]:
import pingouin as pg
from pingouin import rm_anova, sphericity, pairwise_tests
aucprs = final_df[[col for col in final_df  if 'LR AUC-PR'in col]]
aucprs['Index'] = aucprs.index
print(aucprs)
spher = pg.sphericity(data = aucprs.drop(columns = 'Index'))
print(spher.spher)

if spher.spher:
    aov = pg.rm_anova(data = aucprs)
    pg.print_table(aov)
    melted_df = pd.melt(aucprs, id_vars = ['Index'], value_vars = [col for col in aucprs if 'AUC-PR' in col])
    ph = pg.pairwise_tests(data = melted_df, subject = 'Index', dv = 'value', within = ['variable'], parametric = True, padjust = 'bonf')
    print(ph)

   Desikan_coeffs_fs LR AUC-PR  Desikan_cort_thick LR AUC-PR  \
0                     0.927333                      0.922814   
1                     0.939430                      0.906893   
2                     0.919668                      0.924235   
3                     0.919465                      0.908474   
4                     0.868061                      0.820576   

   Desikan_roi_gmv LR AUC-PR  Desikan_vbm LR AUC-PR  Index  
0                   0.922409               0.905378      0  
1                   0.950883               0.936590      1  
2                   0.932232               0.922259      2  
3                   0.921223               0.915508      3  
4                   0.873006               0.858962      4  
True

ANOVA SUMMARY

Source      ddof1    ddof2      F    p_unc    p_GG_corr    ng2    eps  sphericity      W_spher    p_spher
--------  -------  -------  -----  -------  -----------  -----  -----  ------------  ---------  ---------
Within          